
# Task 1 — TimesNet for Traffic Forecasting After Incidents

This notebook implements a compact **TimesNet-style forecasting model** for the supplied TraffiDent-style dataset.

## Dataset structure

Each row has 390 columns:

- 384 input columns = 24 timesteps × 16 features
- 6 target columns = traffic flow at `t+1` through `t+6`

The model predicts `t+1`, `t+3`, and `t+6` under three settings:

1. **TimesNet-General**
2. **TimesNet-Binary**
3. **TimesNet-TypeEmb** with a learnable 16-dimensional incident-type embedding

Each model is evaluated on all test samples and incident-only test samples using MAE, RMSE, and MAPE.

> This is a TraffiDent-inspired adaptation for hourly, pre-windowed data. TimesNet was not one of the original forecasting baselines reported in the TraffiDent forecasting table.


In [2]:

# 1. Imports
from pathlib import Path
import copy, json, random, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
print("PyTorch version:", torch.__version__)


PyTorch version: 2.12.1


## 2. Configuration

In [4]:

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

TRAIN_PATH = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/train_24_6.csv"
VAL_PATH   = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/val_24_6.csv"
TEST_PATH  = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/test_24_6.csv"

N_TIMESTEPS = 24
N_FEATURES = 16
N_INPUT_COLUMNS = 384
EXPECTED_TOTAL_COLUMNS = 390
HORIZONS = [1, 3, 6]
TARGET_INDICES = [0, 2, 5]

TOTAL_FLOW_IDX = 0
IMPACT_SEQUENCE_HOUR_IDX = 5
INCIDENT_TYPE_IDX = 12

# Kept identical to the MICN notebook for fair comparison.
GENERAL_FEATURE_INDICES = [0, 1, 2, 3, 4, 8, 9, 10, 13, 14]

INCIDENT_EMBED_DIM = 16
NUM_INCIDENT_CODES = 9
BATCH_SIZE = 256
MAX_EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
D_MODEL = 64
D_FF = 128
E_LAYERS = 2
TOP_K_PERIODS = 3
NUM_KERNELS = 4
DROPOUT = 0.1
NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

OUTPUT_DIR = Path("timesnet_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)


Device: mps


## 3. Metadata

In [5]:

FEATURE_NAMES = [
    "total_flow", "precipitation", "temperature_2m", "wind_gusts_10m",
    "relative_humidity", "impact_sequence_hour", "is_major_incident",
    "is_holiday", "hour_sin", "hour_cos", "is_weekend", "feat_12",
    "incident_type", "lane_count", "road_functional_hierarchy",
    "distance_to_intersection",
]

INCIDENT_TYPE_MAP = {
    0: "NO_INCIDENT", 1: "CRASH", 2: "BREAKDOWN", 3: "HAZARD",
    4: "ROADWORK", 5: "TRAFFIC_CONTROL", 6: "ADVERSE_WEATHER",
    7: "EVENT", 8: "OTHERS",
}

print("General features:")
for i in GENERAL_FEATURE_INDICES:
    print(f"- {i+1}: {FEATURE_NAMES[i]}")


General features:
- 1: total_flow
- 2: precipitation
- 3: temperature_2m
- 4: wind_gusts_10m
- 5: relative_humidity
- 9: hour_sin
- 10: hour_cos
- 11: is_weekend
- 14: lane_count
- 15: road_functional_hierarchy


## 4. Load headerless train, validation, and test files

In [6]:

def load_matrix(path, split_name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{split_name}: file not found: {path.resolve()}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path, header=None)
    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, header=None)
    elif suffix in [".parquet", ".pq"]:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")
    if df.shape[1] != EXPECTED_TOTAL_COLUMNS:
        raise ValueError(
            f"{split_name}: expected {EXPECTED_TOTAL_COLUMNS} columns, found {df.shape[1]}"
        )
    df = df.apply(pd.to_numeric, errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"{split_name}: missing or non-numeric values found")
    return df

train_raw = load_matrix(TRAIN_PATH, "train")
val_raw = load_matrix(VAL_PATH, "validation")
test_raw = load_matrix(TEST_PATH, "test")
print("Train:", train_raw.shape)
print("Validation:", val_raw.shape)
print("Test:", test_raw.shape)


Train: (603865, 390)
Validation: (206425, 390)
Test: (187105, 390)


## 5. Extract model-ready arrays

In [7]:

def extract_split(df, split_name):
    values = df.to_numpy(dtype=np.float32)
    X_all = values[:, :N_INPUT_COLUMNS].reshape(-1, N_TIMESTEPS, N_FEATURES)
    y_all = values[:, N_INPUT_COLUMNS:]
    y = y_all[:, TARGET_INDICES]
    X_general = X_all[:, :, GENERAL_FEATURE_INDICES]

    impact_at_t = X_all[:, -1, IMPACT_SEQUENCE_HOUR_IDX]
    incident_binary = (impact_at_t >= 0).astype(np.int64)

    incident_type = np.rint(X_all[:, -1, INCIDENT_TYPE_IDX]).astype(np.int64)
    incident_type = np.where(incident_binary == 0, 0, incident_type)

    invalid = ~np.isin(incident_type, np.arange(NUM_INCIDENT_CODES))
    if invalid.any():
        raise ValueError(
            f"{split_name}: invalid incident type values {np.unique(incident_type[invalid])}"
        )

    return {
        "X_general": X_general.astype(np.float32),
        "y": y.astype(np.float32),
        "incident_binary": incident_binary,
        "incident_type": incident_type,
    }

train_data = extract_split(train_raw, "train")
val_data = extract_split(val_raw, "validation")
test_data = extract_split(test_raw, "test")
print("X train:", train_data["X_general"].shape)
print("y train:", train_data["y"].shape)
print("Train incident rate:", train_data["incident_binary"].mean())


X train: (603865, 24, 10)
y train: (603865, 3)
Train incident rate: 0.002974174691363136


## 6. Dataset audit

In [8]:

def audit_split(data, split_name):
    counts = pd.Series(data["incident_type"]).value_counts().sort_index()
    row = {
        "split": split_name,
        "samples": len(data["y"]),
        "incident_rate": float(data["incident_binary"].mean()),
        "x_mean": float(data["X_general"].mean()),
        "x_std": float(data["X_general"].std()),
        "target_mean": float(data["y"].mean()),
        "target_std": float(data["y"].std()),
    }
    for code, label in INCIDENT_TYPE_MAP.items():
        row[f"n_{label}"] = int(counts.get(code, 0))
    return row

audit_df = pd.DataFrame([
    audit_split(train_data, "train"),
    audit_split(val_data, "validation"),
    audit_split(test_data, "test"),
])
display(audit_df)


,split,samples,incident_rate,x_mean,x_std,target_mean,target_std,n_NO_INCIDENT,n_CRASH,n_BREAKDOWN,n_HAZARD,n_ROADWORK,n_TRAFFIC_CONTROL,n_ADVERSE_WEATHER,n_EVENT,n_OTHERS
0,train,603865,0.002974,0.586863,1.531940,0.001636,1.000486,602069,396,629,85,146,167,59,314,0
1,validation,206425,0.002684,0.539749,1.592901,0.222605,1.193259,205871,132,213,25,49,34,6,95,0
2,test,187105,0.003298,0.591528,1.590513,0.142942,1.166631,186488,134,235,34,27,75,14,94,4


## 7. Dataset class and loaders

In [9]:

class TrafficDataset(Dataset):
    def __init__(self, data):
        self.X = torch.tensor(data["X_general"], dtype=torch.float32)
        self.y = torch.tensor(data["y"], dtype=torch.float32)
        self.binary = torch.tensor(data["incident_binary"], dtype=torch.long)
        self.incident_type = torch.tensor(data["incident_type"], dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return {
            "x": self.X[idx], "y": self.y[idx],
            "binary": self.binary[idx],
            "incident_type": self.incident_type[idx],
        }

def make_loader(data, shuffle=False):
    return DataLoader(
        TrafficDataset(data), batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

train_loader = make_loader(train_data, shuffle=True)
val_loader = make_loader(val_data)
test_loader = make_loader(test_data)


## 8. TimesNet-style architecture

In [10]:

def fft_for_period(x, top_k=3):
    # x: [B, T, D]
    xf = torch.fft.rfft(x, dim=1)
    amplitude = torch.abs(xf).mean(dim=0).mean(dim=-1)
    amplitude[0] = 0
    k = min(top_k, max(1, amplitude.shape[0] - 1))
    indices = torch.topk(amplitude, k=k).indices.clamp(min=1)
    periods = (x.shape[1] // indices).clamp(min=1)
    weights = torch.abs(xf).mean(dim=-1)[:, indices]
    return periods, weights

class InceptionBlockV1(nn.Module):
    def __init__(self, in_channels, out_channels, num_kernels=4):
        super().__init__()
        self.kernels = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=2*i+1, padding=i)
            for i in range(num_kernels)
        ])
    def forward(self, x):
        return torch.stack([kernel(x) for kernel in self.kernels], dim=-1).mean(dim=-1)

class TimesBlock(nn.Module):
    def __init__(self, seq_len, d_model, d_ff, top_k=3, num_kernels=4):
        super().__init__()
        self.seq_len = seq_len
        self.top_k = top_k
        self.conv = nn.Sequential(
            InceptionBlockV1(d_model, d_ff, num_kernels),
            nn.GELU(),
            InceptionBlockV1(d_ff, d_model, num_kernels),
        )
    def forward(self, x):
        B, T, D = x.shape
        periods, period_weights = fft_for_period(x, self.top_k)
        outputs = []
        for i in range(len(periods)):
            period = int(periods[i].item())
            if T % period != 0:
                padded_len = ((T // period) + 1) * period
                padding = torch.zeros(B, padded_len - T, D, device=x.device, dtype=x.dtype)
                out = torch.cat([x, padding], dim=1)
            else:
                padded_len = T
                out = x
            out = out.reshape(B, padded_len // period, period, D)
            out = out.permute(0, 3, 1, 2).contiguous()
            out = self.conv(out)
            out = out.permute(0, 2, 3, 1).reshape(B, padded_len, D)
            outputs.append(out[:, :T, :])
        stacked = torch.stack(outputs, dim=-1)
        weights = torch.softmax(period_weights, dim=1).unsqueeze(1).unsqueeze(1)
        weights = weights.repeat(1, T, D, 1)
        return torch.sum(stacked * weights, dim=-1) + x

class TimesNetForecaster(nn.Module):
    def __init__(self, input_dim, mode="general", seq_len=24, d_model=64,
                 d_ff=128, e_layers=2, top_k=3, num_kernels=4,
                 embed_dim=16, dropout=0.1):
        super().__init__()
        if mode not in {"general", "binary", "typeemb"}:
            raise ValueError("Invalid mode")
        self.mode = mode
        self.input_projection = nn.Linear(input_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TimesBlock(seq_len, d_model, d_ff, top_k, num_kernels)
            for _ in range(e_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(e_layers)])
        aux_dim = 0
        if mode == "binary":
            aux_dim = 1
        elif mode == "typeemb":
            self.type_embedding = nn.Embedding(NUM_INCIDENT_CODES, embed_dim, padding_idx=0)
            aux_dim = embed_dim
        self.head = nn.Sequential(
            nn.Linear(d_model + aux_dim, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, len(HORIZONS))
        )
    def forward(self, x, binary, incident_type):
        x = self.dropout(self.input_projection(x))
        for block, norm in zip(self.blocks, self.norms):
            x = norm(block(x))
        representation = x.mean(dim=1)
        if self.mode == "binary":
            representation = torch.cat([representation, binary.float().unsqueeze(1)], dim=1)
        elif self.mode == "typeemb":
            type_vector = self.type_embedding(incident_type)
            type_vector = type_vector * binary.float().unsqueeze(1)
            representation = torch.cat([representation, type_vector], dim=1)
        return self.head(representation)


## 9. Training and early stopping

In [11]:

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    criterion = nn.L1Loss()
    total_loss, total_count = 0.0, 0
    for batch in loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        binary = batch["binary"].to(DEVICE)
        incident_type = batch["incident_type"].to(DEVICE)
        if training:
            optimizer.zero_grad()
        pred = model(x, binary, incident_type)
        loss = criterion(pred, y)
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * len(y)
        total_count += len(y)
    return total_loss / total_count

def train_model(mode):
    seed_everything(SEED)
    model = TimesNetForecaster(
        input_dim=len(GENERAL_FEATURE_INDICES), mode=mode, seq_len=N_TIMESTEPS,
        d_model=D_MODEL, d_ff=D_FF, e_layers=E_LAYERS,
        top_k=TOP_K_PERIODS, num_kernels=NUM_KERNELS,
        embed_dim=INCIDENT_EMBED_DIM, dropout=DROPOUT
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_state, best_val = None, float("inf")
    patience_count, history = 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        with torch.no_grad():
            val_loss = run_epoch(model, val_loader)
        history.append({"epoch": epoch, "train_mae_loss": train_loss, "val_mae_loss": val_loss})
        print(f"{mode:8s} | epoch {epoch:02d} | train {train_loss:.6f} | val {val_loss:.6f}")
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
        if patience_count >= PATIENCE:
            print("Early stopping")
            break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


## 10. Prediction and metrics

In [12]:

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    predictions, targets, binaries, types = [], [], [], []
    for batch in loader:
        x = batch["x"].to(DEVICE)
        binary = batch["binary"].to(DEVICE)
        incident_type = batch["incident_type"].to(DEVICE)
        pred = model(x, binary, incident_type)
        predictions.append(pred.cpu().numpy())
        targets.append(batch["y"].numpy())
        binaries.append(batch["binary"].numpy())
        types.append(batch["incident_type"].numpy())
    return {
        "pred": np.concatenate(predictions), "y": np.concatenate(targets),
        "binary": np.concatenate(binaries), "incident_type": np.concatenate(types)
    }

def safe_mape(y_true, y_pred, epsilon=1e-6):
    mask = np.abs(y_true) > epsilon
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def evaluate_prediction(result, model_name, subset_name, mask=None):
    if mask is None:
        mask = np.ones(len(result["y"]), dtype=bool)
    rows = []
    for j, horizon in enumerate(HORIZONS):
        y_true = result["y"][mask, j]
        y_pred = result["pred"][mask, j]
        rows.append({
            "model": model_name, "subset": subset_name,
            "horizon_hour": horizon, "n_samples": len(y_true),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAPE_percent": safe_mape(y_true, y_pred),
        })
    return pd.DataFrame(rows)


## 11. Run the three TimesNet experiments

In [13]:

EXPERIMENTS = {
    "TimesNet-General": "general",
    "TimesNet-Binary": "binary",
    "TimesNet-TypeEmb": "typeemb",
}
all_metrics, all_histories, all_predictions = [], [], []
trained_models = {}

for model_name, mode in EXPERIMENTS.items():
    print("\n" + "=" * 70)
    print("Training:", model_name)
    start = time.time()
    model, history = train_model(mode)
    elapsed = time.time() - start
    trained_models[model_name] = model
    history["model"] = model_name
    history["training_seconds"] = elapsed
    all_histories.append(history)

    val_result = predict_model(model, val_loader)
    all_metrics.append(evaluate_prediction(val_result, model_name, "validation_all"))
    val_mask = val_result["binary"] == 1
    if val_mask.any():
        all_metrics.append(evaluate_prediction(val_result, model_name, "validation_incident_only", val_mask))

    test_result = predict_model(model, test_loader)
    all_metrics.append(evaluate_prediction(test_result, model_name, "test_all"))
    test_mask = test_result["binary"] == 1
    if test_mask.any():
        all_metrics.append(evaluate_prediction(test_result, model_name, "test_incident_only", test_mask))

    pred_df = pd.DataFrame({
        "sample_index": np.arange(len(test_result["y"])),
        "model": model_name,
        "incident_binary": test_result["binary"],
        "incident_type_code": test_result["incident_type"],
        "incident_type": [INCIDENT_TYPE_MAP[int(v)] for v in test_result["incident_type"]],
    })
    for j, horizon in enumerate(HORIZONS):
        pred_df[f"actual_t+{horizon}"] = test_result["y"][:, j]
        pred_df[f"pred_t+{horizon}"] = test_result["pred"][:, j]
    all_predictions.append(pred_df)

metrics_df = pd.concat(all_metrics, ignore_index=True)
history_df = pd.concat(all_histories, ignore_index=True)
predictions_df = pd.concat(all_predictions, ignore_index=True)
display(metrics_df.sort_values(["subset", "horizon_hour", "MAE"]).reset_index(drop=True))



Training: TimesNet-General
general  | epoch 01 | train 0.234362 | val 0.229918
general  | epoch 02 | train 0.181347 | val 0.211255
general  | epoch 03 | train 0.168663 | val 0.198136
general  | epoch 04 | train 0.161196 | val 0.193556
general  | epoch 05 | train 0.156389 | val 0.186217
general  | epoch 06 | train 0.152676 | val 0.189316
general  | epoch 07 | train 0.149753 | val 0.182450
general  | epoch 08 | train 0.147189 | val 0.183079
general  | epoch 09 | train 0.144948 | val 0.180397
general  | epoch 10 | train 0.142976 | val 0.179847
general  | epoch 11 | train 0.141156 | val 0.178569
general  | epoch 12 | train 0.139602 | val 0.177358
general  | epoch 13 | train 0.137784 | val 0.177671
general  | epoch 14 | train 0.136460 | val 0.175070
general  | epoch 15 | train 0.135164 | val 0.178137
general  | epoch 16 | train 0.133583 | val 0.175912
general  | epoch 17 | train 0.132478 | val 0.175128
general  | epoch 18 | train 0.131284 | val 0.174722
general  | epoch 19 | train 0.130206

,model,subset,horizon_hour,n_samples,MAE,RMSE,MAPE_percent
0,TimesNet-Binary,test_all,1,187105,0.115469,0.286069,177.024019
1,TimesNet-General,test_all,1,187105,0.116448,0.284914,182.308662
2,TimesNet-TypeEmb,test_all,1,187105,0.116549,0.288091,164.609289
3,TimesNet-TypeEmb,test_all,3,187105,0.165852,0.399609,259.169221
4,TimesNet-Binary,test_all,3,187105,0.166129,0.398045,296.548557
5,TimesNet-General,test_all,3,187105,0.168539,0.400476,375.824094
6,TimesNet-Binary,test_all,6,187105,0.223390,0.534156,426.348877
7,TimesNet-TypeEmb,test_all,6,187105,0.223694,0.528291,496.664190
8,TimesNet-General,test_all,6,187105,0.226741,0.534023,655.109787
9,TimesNet-TypeEmb,test_incident_only,1,617,0.098211,0.273322,35.991079


## 12. Marginal value of incident information

In [14]:

def build_delta_table(metrics, subset):
    selected = metrics[metrics["subset"] == subset]
    pivot = selected.pivot_table(index="horizon_hour", columns="model", values="MAE", aggfunc="first")
    pivot["Delta1_General_minus_Binary"] = pivot["TimesNet-General"] - pivot["TimesNet-Binary"]
    pivot["Delta2_Binary_minus_TypeEmb"] = pivot["TimesNet-Binary"] - pivot["TimesNet-TypeEmb"]
    return pivot.reset_index()

delta_all = build_delta_table(metrics_df, "test_all")
delta_incident = build_delta_table(metrics_df, "test_incident_only")
print("All test samples")
display(delta_all)
print("Incident-only test samples")
display(delta_incident)


All test samples


model,horizon_hour,TimesNet-Binary,TimesNet-General,TimesNet-TypeEmb,Delta1_General_minus_Binary,Delta2_Binary_minus_TypeEmb
0,1,0.115469,0.116448,0.116549,0.000979,-0.001080
1,3,0.166129,0.168539,0.165852,0.002409,0.000277
2,6,0.223390,0.226741,0.223694,0.003351,-0.000305


Incident-only test samples


model,horizon_hour,TimesNet-Binary,TimesNet-General,TimesNet-TypeEmb,Delta1_General_minus_Binary,Delta2_Binary_minus_TypeEmb
0,1,0.099114,0.101418,0.098211,0.002303,0.000903
1,3,0.145765,0.156111,0.155600,0.010346,-0.009835
2,6,0.204139,0.207062,0.213089,0.002923,-0.008950


## 13. Final result table

In [15]:

final_table = (
    metrics_df[metrics_df["subset"].isin(["test_all", "test_incident_only"])]
    .pivot_table(
        index=["model", "subset"], columns="horizon_hour",
        values=["MAE", "RMSE", "MAPE_percent"], aggfunc="first"
    )
    .round(4)
)
display(final_table)


MAE                 MAPE_percent  \
horizon_hour                              1       3       6            1   
model            subset                                                    
TimesNet-Binary  test_all            0.1155  0.1661  0.2234     177.0240   
                 test_incident_only  0.0991  0.1458  0.2041      39.3364   
TimesNet-General test_all            0.1164  0.1685  0.2267     182.3087   
                 test_incident_only  0.1014  0.1561  0.2071      43.2282   
TimesNet-TypeEmb test_all            0.1165  0.1659  0.2237     164.6093   
                 test_incident_only  0.0982  0.1556  0.2131      35.9911   

                                                           RMSE          \
horizon_hour                                3         6       1       3   
model            subset                                                   
TimesNet-Binary  test_all            296.5486  426.3489  0.2861  0.3980   
                 test_incident_only   29.0655  139.4065  0.2767  0.3580   
TimesNet-General test_all            375.8241  655.1098  0.2849  0.4005   
                 test_incident_only   30.3457  135.7568  0.2899  0.4276   
TimesNet-TypeEmb test_all            259.1692  496.6642  0.2881  0.3996   
                 test_incident_only   30.0006  135.3008  0.2733  0.3898   

                                             
horizon_hour                              6  
model            subset                      
TimesNet-Binary  test_all            0.5342  
                 test_incident_only  0.5061  
TimesNet-General test_all            0.5340  
                 test_incident_only  0.5410  
TimesNet-TypeEmb test_all            0.5283  
                 test_incident_only  0.5425

## 14. Save outputs

In [ ]:

metrics_df.to_csv(OUTPUT_DIR / "timesnet_metrics.csv", index=False)
history_df.to_csv(OUTPUT_DIR / "timesnet_training_history.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "timesnet_test_predictions.csv", index=False)
delta_all.to_csv(OUTPUT_DIR / "timesnet_delta_test_all.csv", index=False)
delta_incident.to_csv(OUTPUT_DIR / "timesnet_delta_test_incident_only.csv", index=False)
audit_df.to_csv(OUTPUT_DIR / "timesnet_dataset_audit.csv", index=False)

for model_name, model in trained_models.items():
    safe_name = model_name.lower().replace("-", "_")
    torch.save(model.state_dict(), OUTPUT_DIR / f"{safe_name}.pt")

config = {
    "seed": SEED,
    "horizons": HORIZONS,
    "general_feature_indices_zero_based": GENERAL_FEATURE_INDICES,
    "general_feature_names": [FEATURE_NAMES[i] for i in GENERAL_FEATURE_INDICES],
    "incident_embedding_dim": INCIDENT_EMBED_DIM,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "d_model": D_MODEL,
    "d_ff": D_FF,
    "e_layers": E_LAYERS,
    "top_k_periods": TOP_K_PERIODS,
    "num_kernels": NUM_KERNELS,
    "dropout": DROPOUT,
    "incident_type_mapping": INCIDENT_TYPE_MAP,
}
with open(OUTPUT_DIR / "timesnet_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print("Saved to:", OUTPUT_DIR.resolve())



## Interpretation guidance

Use **Section 13** as the main table and **Section 12** for marginal incident value.

- Positive `Delta1`: Binary improves over General.
- Positive `Delta2`: TypeEmb improves over Binary.
- Focus primarily on `test_incident_only` for post-incident forecasting.
- Because targets are Z-score normalized, MAE and RMSE are in normalized units. MAPE can be unstable near zero.
